# SQL analytics với SQLite

## Mục tiêu
- Viết SELECT/WHERE/GROUP BY/JOIN rõ ràng.
- Dùng CTE và window function cho bài toán phân tích.
- Luôn kiểm tra grain trước khi JOIN để tránh nhân bản doanh thu.

In [ ]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(':memory:')
customers = pd.DataFrame({'customer_id': [1, 2, 3], 'city': ['Hanoi', 'Da Nang', 'Hanoi']})
orders = pd.DataFrame({'order_id': [101, 102, 103, 104], 'customer_id': [1, 1, 2, 3], 'amount': [200, 350, 500, 120]})
customers.to_sql('customers', connection, index=False)
orders.to_sql('orders', connection, index=False)

In [ ]:
query = '''
SELECT c.city, COUNT(o.order_id) AS orders, SUM(o.amount) AS revenue
FROM customers AS c
JOIN orders AS o USING (customer_id)
GROUP BY c.city
ORDER BY revenue DESC
'''
pd.read_sql_query(query, connection)

## Window function
`GROUP BY` gộp các dòng thành một dòng mỗi nhóm. Window function giữ nguyên từng dòng và thêm metric tính trong nhóm: rất hữu ích cho rank, running total, share of total.

In [ ]:
query = '''
WITH customer_revenue AS (
  SELECT customer_id, SUM(amount) AS revenue FROM orders GROUP BY customer_id
)
SELECT customer_id, revenue,
       RANK() OVER (ORDER BY revenue DESC) AS revenue_rank,
       ROUND(100.0 * revenue / SUM(revenue) OVER (), 1) AS pct_of_total
FROM customer_revenue
'''
pd.read_sql_query(query, connection)

## Tự luyện
1. Viết query tìm customer có tổng spend > 300.
2. Thêm cột ngày vào orders và tính running revenue.
3. Nêu grain của `customers`, `orders` và kết quả sau JOIN.

**Checkpoint:** Bạn kiểm tra row count trước/sau JOIN, đặc biệt khi join nhiều bảng.